# The Ninefold Tide — E101 camera/Grease Pencil production stage v04

Reuses the private original v09r1 source SHA-256 and official Blender 4.5.15 archive. Corrects Blender 4.5's extra default Grease Pencil layer **only in an extracted working script**, producing the canonical nine named empty drawing layers per Naya/Kito/Iri. Original source, stage mechanism and art remain untouched. Passes technical stage checks only, not creative approval.


In [ ]:
from pathlib import Path
import datetime,hashlib,json,os,sys,traceback,zipfile

ROOT=Path("/kaggle/working/ninefold_v09_private_scene")
ROOT.mkdir(parents=True,exist_ok=True)
manifest=ROOT/"scene_build_status.json"
status={
  "project":"The Ninefold Tide","episode":101,"scene":"CAM03B",
  "purpose":"Real Blender 4.5 editable scene from private v09r1 source kit, NOT final drawn animation",
  "status":"INITIALIZING","blender_stage_generated":False,
  "original_character_drawings_approved":False,
  "artistic_animation_approved":False,
  "gpu_allocated":False,
  "uses_existing_private_cached_blender":True,
  "source_kit_downloaded_to_phone":False,
  "canon":"https://docs.google.com/document/d/1TnnN33aXBXqhjNeAPzdwZeduLPOzFajGCDYL8NwkSBY/edit",
  "source_drive":"https://drive.google.com/file/d/1fOMgzjCH2CYGj_9KeveiQQdPeR1vF-g_/view",
  "started_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
  "errors":[]}
def save():
    manifest.write_text(json.dumps(status,indent=2)+"\n",encoding="utf8")
save()

PRIVATE_INPUT=Path("/kaggle/input")
KIT_NAME="Ninefold_S1_Private_Kaggle_Kit_v09r1.zip"
CACHE_NAME="blender-4.5.15-linux-x64.tar.xz"
kits=list(PRIVATE_INPUT.rglob(KIT_NAME)) if PRIVATE_INPUT.exists() else []
# Kaggle unpacks ZIPs uploaded as .zip even inside private datasets. The
# second secure upload stores EXACT original ZIP bytes with a .bin extension.
# Copy only this 1.9MB verified original into our private notebook workspace,
# restoring the original name for the unchanged canonical Blender builders.
if not kits:
    opaque=list(PRIVATE_INPUT.rglob("Ninefold_S1_Private_Kaggle_Kit_v09r1.bin")) if PRIVATE_INPUT.exists() else []
    if len(opaque)==1:
        import shutil
        target=Path("/kaggle/working")/KIT_NAME
        with opaque[0].open("rb") as source, target.open("wb") as dst:
            shutil.copyfileobj(source,dst,1024*1024)
        kits=[target]
        print("Restored 1.9MB private original archive from opaque Kaggle input",flush=True)
caches=list(PRIVATE_INPUT.rglob(CACHE_NAME)) if PRIVATE_INPUT.exists() else []
status["private_kit_count"]=len(kits)
status["private_blender_cache_count"]=len(caches)

if len(kits)!=1:
    status["status"]="INPUT_REQUIRED_PRIVATE_1_9MB_DRIVE_KIT"
    status["errors"].append(
       "In Kaggle notebook Add Input, attach the PRIVATE source ZIP named "+KIT_NAME+
       " from the project's Google Drive ART v09r1 folder. This cannot be inferred from source code.")
    save()
    print("NINEFOLD_V09_KIT_NOT_ATTACHED",json.dumps(status,indent=2),flush=True)
elif len(caches)!=1:
    status["status"]="PRIVATE_BLENDER_CACHE_INPUT_MISSING"
    status["errors"].append(
      "Attach the private output of aighewieghosa/ninefold-blender-4-5-15-cache as kernel source. "
      "No new 378MB download is necessary.")
    save()
    print("NINEFOLD_V09_BLENDER_CACHE_INPUT_NOT_ATTACHED",json.dumps(status,indent=2),flush=True)
else:
    try:
        kit=kits[0]
        if not 1_000_000<kit.stat().st_size<8_000_000:
            raise RuntimeError("Unexpected v09r1 source ZIP size; stop instead of executing unknown assets")
        h=hashlib.sha256()
        with open(kit,"rb") as fd:
            for block in iter(lambda:fd.read(1024*1024),b""):
                h.update(block)
        status["private_source_kit_sha256"]=h.hexdigest()
        status["private_source_kit_bytes"]=kit.stat().st_size
        with zipfile.ZipFile(kit) as archive:
            if archive.testzip():raise RuntimeError("v09r1 private source archive checksum/integrity failed")
            notebook=json.loads(archive.read("Ninefold_S1_Kaggle_AllStages_v09.ipynb"))
        cells=[x for x in notebook["cells"] if x["cell_type"]=="code"]
        if len(cells)<4:raise RuntimeError("v09 source notebook did not contain four source/build stages")
        source=[''.join(x["source"]) for x in cells]
        # The original art v09r1 package was renamed when revised, while the
        # embedded notebook still searched for the old v09 name.
        old="KIT_NAME='Ninefold_S1_Private_Kaggle_Kit_v09.zip'"
        replacement="KIT_NAME='Ninefold_S1_Private_Kaggle_Kit_v09r1.zip'"
        if old not in source[0]:
            raise RuntimeError("Unexpected v09 source format; cannot safely patch the known filename")
        source[0]=source[0].replace(old,replacement)
        # The verified Blender .tar.xz already lives in private Kaggle INPUT.
        # Reject a fallback 378MB re-download if attachment is incomplete.
        source[1]=source[1].replace("ALLOW_REMOTE_BLENDER_DOWNLOAD=True",
                                    "ALLOW_REMOTE_BLENDER_DOWNLOAD=False")
        namespace={"__name__":"__main__"}
        names=("01_VERIFY_PRIVATE_V07_V08_ART","02_USE_CACHED_OFFICIAL_BLENDER",
               "03_BUILD_V05_V07_V08_ORIGINAL_SCENE","04_SAVE_REAL_3D_SCENE_REPORT")
        for name,block in zip(names,source[:4]):
            if name=="03_BUILD_V05_V07_V08_ORIGINAL_SCENE":
                # Blender 4.5 creates one default GP layer even for type=EMPTY;
                # the original v07 script adds nine more, yielding TEN and
                # violating the 9-layer quality gate. Fix only extracted working
                # source, NEVER modify the private canonical ZIP.
                gp_file=namespace["S7"]/"BUILD_2D_GREASE_PENCIL_WORKSPACE_v07.py"
                original_script=gp_file.read_text(encoding="utf8")
                old_marker="    for layer_name in LAYER_NAMES:"
                if original_script.count(old_marker)!=1:
                    raise RuntimeError("Unexpected v07 original Grease Pencil layer loop")
                patch=("    # Blender 4.5 initializes an extra default layer.\\n"
                       "    existing=list(drawing.data.layers)\\n"
                       "    if len(existing)==1 and existing[0].name not in LAYER_NAMES:\\n"
                       "        drawing.data.layers.remove(existing[0])\\n"
                       "    elif existing:\\n"
                       "        raise RuntimeError('Unexpected initial GP layers: '+str([p.name for p in existing]))\\n")
                working_script=original_script.replace(old_marker,patch+old_marker)
                gp_file.write_text(working_script,encoding="utf8")
                status["gp_default_layer_workaround"]="Blender 4.5 default layer removed only from extracted working script"
                print("NINEFOLD_SAFE_GP_LAYER_PATCH_APPLIED",flush=True)
            print("NINEFOLD_STAGE_STARTED",name,flush=True)
            compiled=compile(block,name,"exec")
            exec(compiled,namespace)
            status["last_completed_stage"]=name
            save()
            print("NINEFOLD_STAGE_COMPLETED",name,flush=True)
        source_report=namespace["report"]
        final_scene=namespace["blend8"]
        if not Path(final_scene).is_file() or Path(final_scene).stat().st_size<4096:
            raise RuntimeError("Blender stage did not create a meaningful .blend source")
        if not Path(source_report).exists():
            raise RuntimeError("No physical-camera v09 source report")
        report=json.loads(Path(source_report).read_text(encoding="utf8"))
        if report.get("release_approved") is not False:
            raise RuntimeError("Refuse to label unpainted technical stage as finished art")
        status["status"]="BLENDER_PHYSICAL_SCENE_BUILD_EXECUTED_ART_UNAPPROVED"
        status["blender_stage_generated"]=True
        status["blend_path"]=str(final_scene)
        status["blend_size_bytes"]=Path(final_scene).stat().st_size
        status["source_report_path"]=str(source_report)
    except Exception as exc:
        status["status"]="FAILED_PRESERVE_LOGS"
        status["errors"].append(type(exc).__name__+": "+str(exc)[:1000])
        (ROOT/"build_error_traceback.txt").write_text(traceback.format_exc()[-6000:],encoding="utf8")
    finally:
        status["ended_utc"]=datetime.datetime.now(datetime.timezone.utc).isoformat()
        save()
        print("NINEFOLD_V09_REAL_SCENE_STATUS",json.dumps({
            "status":status["status"],
            "blender_scene_generated":status["blender_stage_generated"],
            "source_kit_bytes":status.get("private_source_kit_bytes"),
            "last_completed_stage":status.get("last_completed_stage"),
            "errors":status["errors"]},indent=2),flush=True)
